# KINZ Secure Commerce Hub — Exploratory Data Analysis

**Author:** Nassim K. (Business Analyst, Co-Founder @ KINZ)  
**Data:** `data/processed/sales_2023_2024.csv` (4,500 orders, ~13.7k line items, 2023-01 → 2024-12)  
**Catalog:** `data/raw/products.csv` (30 real KINZ SKUs scraped from kinzoils.com)

This notebook answers three business questions for the KINZ leadership team:

1. **Customer Lifetime Value by Channel** — which acquisition channel produces the highest CLV?
2. **High-Margin SKU Concentration** — which categories drive margin vs. revenue, and where should we focus Q4 promo spend?
3. **Channel Risk** — are some channels exhibiting high discount rates or return signals that warrant a marketing-strategy review?

All findings are backed by reproducible code and exported charts in `analytics/charts/`.

In [ ]:
import os
import sys
from pathlib import Path

# ==========================================
# 1. ENVIRONMENT DETECTION & AUTO-CLONING
# ==========================================
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    # Scenario: Google Colab (Magic Link or manual open)
    REPO_URL = "https://github.com/nassim0014/kinz-secure-commerce-hub.git"
    REPO_NAME = "kinz-secure-commerce-hub"
    COLAB_ROOT = Path('/content') / REPO_NAME
    
    if not COLAB_ROOT.exists():
        print(f"\U0001f680 Colab environment detected. Cloning repository...")
        !git clone {REPO_URL} /content/{REPO_NAME}
    else:
        print(f"\u2705 Repository already exists in Colab.")
    ROOT = COLAB_ROOT
else:
    # Scenario: Local (Anaconda, VS Code, PyCharm, Codespaces, Terminal)
    current_dir = Path.cwd()
    ROOT = None
    
    # Bounded traversal: Check up to 6 directory levels up. 
    # This mathematically prevents infinite loops.
    for _ in range(6):
        # Look for a unique marker that only exists in the project root
        if (current_dir / 'data' / 'processed' / 'sales_2023_2024.csv').exists():
            ROOT = current_dir
            break
        # Stop if we hit the absolute filesystem root (e.g., C:\\ or /)
        if current_dir == current_dir.parent:
            break
        current_dir = current_dir.parent
        
    if ROOT is None:
        raise FileNotFoundError(
            "\u274c Could not locate the project root (missing 'data/processed/sales_2023_2024.csv').\n"
            "If running locally, please ensure you have cloned the repo and opened this notebook from within the project directory."
        )

# ==========================================
# 2. FINALIZE PATHS & SETUP
# ==========================================
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'

print(f"\u2705 Environment initialized. Root directory: {ROOT}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# Font fallback so any CJK / emoji-ish glyphs still render
try:
    fm.fontManager.addfont('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
except Exception:
    pass
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

sns.set_theme(style='whitegrid', palette='muted')
OUT = ROOT / 'analytics' / 'charts'
OUT.mkdir(parents=True, exist_ok=True)

products = pd.read_csv(DATA / 'raw' / 'products.csv')
sales    = pd.read_csv(DATA / 'processed' / 'sales_2023_2024.csv')
customers= pd.read_csv(DATA / 'processed' / 'customers.csv')

sales['order_date'] = pd.to_datetime(sales['order_date'])
print(f'Products:   {len(products):>5}  SKUs')
print(f'Sales:      {len(sales):>5}  line items across {sales.order_id.nunique()} orders')
print(f'Customers:  {len(customers):>5}  unique accounts')
print(f'Date range: {sales.order_date.min().date()}  →  {sales.order_date.max().date()}')

## 1. Customer Lifetime Value by Channel

**Definition.** CLV per channel = (sum of `order_total_tnd` for that channel) ÷ (number of unique customers acquired via that channel).  
**Why it matters.** Channels with high CLV deserve dedicated account management; channels with low CLV should be optimized or sunset.

In [ ]:
clv = (
    sales.groupby(['channel', 'customer_id'])
         .agg(total_spend=('order_total_tnd', 'sum'),
              orders=('order_id', 'nunique'))
         .reset_index()
         .groupby('channel')
         .agg(clv_per_customer=('total_spend', 'mean'),
              avg_orders=('orders', 'mean'),
              customers=('customer_id', 'nunique'),
              total_revenue=('total_spend', 'sum'))
         .sort_values('clv_per_customer', ascending=False)
         .round(2)
)
clv

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
clv_sorted = clv.sort_values('clv_per_customer', ascending=True)
bars = ax.barh(clv_sorted.index, clv_sorted.clv_per_customer, color='#0f766e')
ax.set_xlabel('CLV per customer (TND)')
ax.set_title('KINZ — Customer Lifetime Value by Acquisition Channel', fontsize=13, fontweight='bold')
for bar, val in zip(bars, clv_sorted.clv_per_customer):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, f'{val:,.0f} TND', va='center', fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.savefig(OUT / 'clv_by_channel.png', dpi=160)
plt.show()

### Insight 1 — CLV by Channel

**B2B channels (Pharmacy, Spa, Reseller) deliver 3–4× the CLV of B2C web.** A dedicated B2B account manager is justified once the B2B base exceeds ~80 customers (we currently have ~110).

Among B2C, the **Instagram channel lags Facebook and Web** — likely because Instagram promotions skew toward one-time discount hunters. Recommendation: sunset Instagram-only promo codes in Q3 and reallocate that budget to Facebook retargeting.

## 2. High-Margin SKU Concentration

Vegetable Oils (esp. prickly pear seed oil) are KINZ's heritage product. We compare each category's contribution to **revenue** vs. **gross margin**.

In [ ]:
# Join sales with product cost to compute margin per line
merged = sales.merge(products[['product_id','cost_tnd','category']], on='product_id', how='left', suffixes=('','_p'))
merged['cogs'] = merged['cost_tnd'] * merged['quantity']
merged['line_margin'] = merged['line_total_tnd'] - merged['cogs']

cat = (
    merged.groupby('category_p' if 'category_p' in merged.columns else 'category')
          .agg(revenue=('line_total_tnd','sum'),
               margin=('line_margin','sum'),
               units=('quantity','sum'))
          .sort_values('revenue', ascending=False)
)
cat['margin_pct'] = (cat.margin / cat.revenue * 100).round(2)
cat['revenue_share_pct'] = (cat.revenue / cat.revenue.sum() * 100).round(2)
cat

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
x = np.arange(len(cat))
w = 0.38
ax.bar(x - w/2, cat.revenue_share_pct, w, label='Revenue share %', color='#0f766e')
ax.bar(x + w/2, cat.margin_pct,       w, label='Gross margin %',   color='#f59e0b')
ax.set_xticks(x)
ax.set_xticklabels(cat.index, rotation=20, ha='right')
ax.set_ylabel('%')
ax.set_title('KINZ — Revenue Share vs. Gross Margin by Category', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.spines[['top','right']].set_visible(False)
plt.savefig(OUT / 'margin_vs_revenue.png', dpi=160)
plt.show()

### Insight 2 — Margin vs. Revenue Mix

**Vegetable Oils deliver the highest gross margin %**, but **Gift Sets drive the highest absolute revenue**. The two are not in competition — they serve different jobs (oils = daily ritual, gift sets = Q4 gifting).

Recommendation: build a Q4 bundle that pairs a hero oil (Prickly Pear Seed Oil) with a mid-tier gift set at a 5–8% bundle discount. This should lift basket size without eroding the oil's premium margin.

## 3. Channel Risk — Discount & Basket Signals

Discount rate per channel + average basket size. Channels with high discount + small basket are leaking margin.

In [ ]:
risk = (
    sales.groupby('channel')
         .agg(avg_discount=('discount_rate','mean'),
              avg_basket=('order_total_tnd','mean'),
              orders=('order_id','nunique'))
         .round(3)
         .sort_values('avg_discount', ascending=False)
)
risk['avg_discount_pct'] = (risk.avg_discount * 100).round(1)
risk

In [ ]:
# Channel-aware color palette so the legend is meaningful
PALETTE = {
    'B2C Web':       '#0f766e',  # olive/teal
    'B2C Facebook':  '#3b82f6',  # blue
    'B2C Instagram': '#ec4899',  # pink (danger brand)
    'B2B Pharmacy':  '#8a5e21',  # brown
    'B2B Spa':       '#a16207',  # amber-brown
    'B2B Reseller':  '#1e3a8a',  # dark blue
}
risk_plot = risk.reset_index().sort_values('orders', ascending=False).reset_index(drop=True)
colors = [PALETTE.get(c, '#666666') for c in risk_plot['channel']]

import matplotlib.ticker as mtick
import matplotlib.lines as mlines

fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)

# Bubble scatter — sizes scaled from order count
sizes = (risk_plot['orders'] / risk_plot['orders'].max() * 1400) + 200
ax.scatter(
    risk_plot['avg_basket'], risk_plot['avg_discount_pct'],
    s=sizes, c=colors, alpha=0.75, edgecolors='white', linewidth=1.5, zorder=3,
)

# Quadrant guides
mean_basket = risk_plot['avg_basket'].mean()
mean_disc   = risk_plot['avg_discount_pct'].mean()
ax.axhline(mean_disc,   ls='--', c='#94a3b8', lw=1.2, alpha=0.7, zorder=1)
ax.axvline(mean_basket, ls='--', c='#94a3b8', lw=1.2, alpha=0.7, zorder=1)

# Axis bounds (with headroom so labels are not clipped)
x_max = risk_plot['avg_basket'].max() * 1.15
y_max = risk_plot['avg_discount_pct'].max() * 1.20
ax.set_xlim(-x_max * 0.05, x_max)
ax.set_ylim(-y_max * 0.05, y_max)

# Quadrant labels
quad_kwargs = dict(fontsize=10, alpha=0.55, ha='center', va='center', style='italic', zorder=0)
ax.text(mean_basket / 2, y_max * 0.92,
        '⚠ DANGER ZONE\nHigh discount · Small basket\n→ Margin leak',
        color='#dc2626', fontweight='bold', **{**quad_kwargs, 'alpha': 0.7})
ax.text((mean_basket + x_max) / 2, y_max * 0.92,
        'B2B WHOLESALE\nHigh discount · Large basket\n→ Negotiated volume rebates',
        color='#1e40af', **quad_kwargs)
ax.text(mean_basket / 2, y_max * 0.10,
        'B2C ORGANIC\nLow discount · Small basket\n→ Healthy D2C',
        color='#0f766e', **quad_kwargs)
ax.text((mean_basket + x_max) / 2, y_max * 0.10,
        'IDEAL\nLow discount · Large basket\n→ Scale this',
        color='#16a34a', **quad_kwargs)

# Smart label placement (manual offsets per channel to avoid overlap)
LABEL_OFFSETS = {
    'B2C Web':       ( 15,  10, 'left'),
    'B2C Facebook':  ( 15, -14, 'left'),
    'B2C Instagram': ( 15,  12, 'left'),
    'B2B Pharmacy':  (-25,  14, 'right'),
    'B2B Spa':       ( 15, -14, 'left'),
    'B2B Reseller':  (-25,  14, 'right'),
}
for _, row in risk_plot.iterrows():
    dx, dy, ha = LABEL_OFFSETS.get(row['channel'], (15, 10, 'left'))
    ax.annotate(
        f"{row['channel']}\n"
        f"  {row['avg_basket']:.0f} TND basket · {row['avg_discount_pct']:.1f}% disc · {row['orders']} orders",
        xy=(row['avg_basket'], row['avg_discount_pct']),
        xytext=(dx, dy), textcoords='offset points',
        ha=ha, va='center', fontsize=9, color='#1f2937',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#cbd5e1', lw=0.6, alpha=0.9),
        zorder=5,
    )

ax.set_xlabel('Average basket size (TND)  →', fontsize=11, color='#374151')
ax.set_ylabel('Average discount rate (%)  →', fontsize=11, color='#374151')
ax.set_title('KINZ — Channel Risk Map: Discount vs. Basket Size',
             fontsize=14, fontweight='bold', color='#1c1407', pad=14)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Bubble-size legend (top-right)
legend_handles = []
for n_orders, label in [(400, '~400 orders'), (800, '~800 orders'), (1400, '~1400 orders')]:
    size = (n_orders / risk_plot['orders'].max() * 1400) + 200
    legend_handles.append(
        mlines.Line2D([], [], marker='o', color='w',
                      markerfacecolor='#9ca3af', markeredgecolor='white',
                      markersize=np.sqrt(size), alpha=0.75, label=label)
    )
size_legend = ax.legend(
    handles=legend_handles, title='Order volume (bubble size)',
    loc='upper right', bbox_to_anchor=(0.99, 0.99),
    frameon=True, framealpha=0.95, edgecolor='#cbd5e1',
    labelspacing=2.2, borderpad=1.0,
)
ax.add_artist(size_legend)

# Channel-color legend (bottom-right)
color_handles = [
    mlines.Line2D([], [], marker='o', color='w', markerfacecolor=c, markeredgecolor='white',
                  markersize=10, label=ch)
    for ch, c in PALETTE.items()
]
ax.legend(
    handles=color_handles, title='Channel',
    loc='lower right', bbox_to_anchor=(0.99, 0.01),
    frameon=True, framealpha=0.95, edgecolor='#cbd5e1',
)

ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.25, linestyle=':', zorder=0)
plt.savefig(OUT / 'channel_risk.png', dpi=160, facecolor='white')
plt.show()


### Insight 3 — Channel Risk Map

**B2C Instagram sits squarely in the danger zone** — its ~14% average discount rate is the highest of any B2C channel (vs. ~1.5% on Web and ~5% on Facebook), while its average basket (61 TND) is the smallest. The bubble size shows it's also the second-largest B2C channel by order volume, so the leak is material, not a rounding error.

By contrast, the B2B channels (Pharmacy, Spa, Reseller) sit in the top-right quadrant: high discount, but baskets are 30–40× larger, so the volume rebates are economically justified. B2C Web is the healthiest channel — low discount, mid-sized baskets, high order volume.

Recommended actions for Q3:
1. Replace Instagram-only promo codes with a 2-step landing page that captures email (raises CAC payback through lifecycle email).
2. Reallocate 30% of Instagram ad spend to Facebook retargeting — Facebook's basket is 2× larger at one-third the discount rate.
3. Re-invest the saved budget into a Prickly Pear Seed Oil hero video — high-margin anchor product that performs well on Web.


## Summary

| # | Insight | Recommended Action |
|---|---------|---------------------|
| 1 | B2B CLV is 3–4× B2C CLV | Hire a dedicated B2B account manager in Q3 |
| 2 | Vegetable Oils = margin leader; Gift Sets = revenue leader | Build a Q4 hero bundle pairing the two |
| 3 | Instagram channel is leaking margin (high discount, low basket) | Replace promo codes with email-capture landing page; reallocate 30% budget to Facebook retargeting |

Charts saved to `analytics/charts/`.